# 02B Incremental → Append Demo

Run one inventory day at a time to bootstrap an Incremental Read, append new movements, and then demonstrate a no-change run.


## Before you begin

- Import this notebook beside `00_env_config` and attach the same Fabric Environment.
- Upload `templates/DemoData` to the Bronze Lakehouse `Files/Demo` location as described in Guided Demo Step 00B.
- Run Day 1, then Day 2, then Day 3. Do not skip or reorder days.
- Re-running the same day is safe: Bronze is rebuilt by `movement_id`, and the governed Incremental → Append lifecycle processes only unconsumed rows.


# 0. Environment


In [ ]:
%run 00_env_config


In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    orchestrate_read,
    orchestrate_write,
    read_lakehouse_csv,
    read_lakehouse_table,
    resolve_table_id,
    widget_select_data_contract,
    write_lakehouse_table,
)


# 1. Choose the demo day

Start with `1`. After a successful complete run, change only this value to `2`, rerun the notebook from the top, then repeat with `3`. Day 3 intentionally adds no source rows.


In [ ]:
DEMO_DAY = 1

if DEMO_DAY not in {1, 2, 3}:
    raise ValueError("DEMO_DAY must be 1, 2, or 3.")


# 2. Read the CSV batch and stage Bronze


In [ ]:
inventory_files = {
    1: "Demo/incremental_inventory/inventory_day1.csv",
    2: "Demo/incremental_inventory/inventory_day2.csv",
}

if DEMO_DAY in inventory_files:
    batch_df = read_lakehouse_csv(
        inventory_files[DEMO_DAY],
        store="Bronze",
        spark_session=spark,
        header=True,
        inferSchema=True,
    )
else:
    # Day 3 has no file by design. Use an empty frame with the managed source schema.
    batch_df = read_lakehouse_table(
        table_name="inventory_movements",
        store="Bronze",
        schema="demo",
        spark_session=spark,
    ).limit(0)


In [ ]:
if DEMO_DAY == 1:
    bronze_df = batch_df.dropDuplicates(["movement_id"])
else:
    existing_bronze_df = read_lakehouse_table(
        table_name="inventory_movements",
        store="Bronze",
        schema="demo",
        spark_session=spark,
    )
    bronze_df = existing_bronze_df.unionByName(batch_df).dropDuplicates(["movement_id"])

write_lakehouse_table(
    bronze_df,
    table_name="inventory_movements",
    store="Bronze",
    schema="demo",
    mode="overwrite",
)

print(f"Bronze inventory rows after Day {DEMO_DAY}: {bronze_df.count()}")


# 3. Data Contract

Select contracts when they exist. To publish the Append target, leave it unselected in Development or select **Enforce**; **Validate** evaluates without publishing.


In [ ]:
CONTRACTS = widget_select_data_contract(spark_session=spark)


# 4. Governed Incremental Read

The target identity makes watermark progress specific to this source-to-target relationship. The public `read_parameters` API selects `modified_datetime` as the watermark.


In [ ]:
sources = {}


In [ ]:
target_table_id = resolve_table_id(
    store="Silver",
    schema="demo",
    table_name="inventory_movements_incremental",
)

source = orchestrate_read(
    name="inventory_movements",
    store="Bronze",
    schema="demo",
    table_name="inventory_movements",
    read_mode="incremental",
    read_parameters={"watermark_column": "modified_datetime"},
    target_table_id=target_table_id,
    spark_session=spark,
)
sources["inventory_movements"] = source


# 5. PySpark Transform


In [ ]:
transformed_df = (
    sources["inventory_movements"]["dataframe"]
    .withColumn("movement_date", F.to_date("modified_datetime"))
    .withColumn("quantity_in", F.greatest(F.col("quantity_change"), F.lit(0)))
    .withColumn("quantity_out", F.greatest(-F.col("quantity_change"), F.lit(0)))
)


# 6. Governed Append Write

Always call the Write orchestrator, including on Day 3. FabricOps owns empty incremental-scope handling and commits watermark progress only after successful publication.


In [ ]:
writes = {}


In [ ]:
writes["inventory_movements_incremental"] = orchestrate_write(
    transformed_df,
    name="inventory_movements_incremental",
    sources=[sources["inventory_movements"]],
    store="Silver",
    schema="demo",
    table_name="inventory_movements_incremental",
    write_mode="append",
    contracts=CONTRACTS,
    repartition_by=None,
    spark_session=spark,
)


# 7. Inspect results


In [ ]:
incremental_target_df = read_lakehouse_table(
    table_name="inventory_movements_incremental",
    store="Silver",
    schema="demo",
    spark_session=spark,
)

print("Incremental scope:", sources["inventory_movements"]["scope"])
print("Silver rows:", incremental_target_df.count())
display(incremental_target_df.orderBy("modified_datetime", "movement_id"))
display(
    incremental_target_df.groupBy("product_id")
    .agg(F.sum("quantity_change").alias("inventory_balance"))
    .orderBy("product_id")
)


## Expected results and reset

| Day | New source rows | Expected Silver rows |
|---:|---:|---:|
| 1 | 8 | 8 |
| 2 | 4 | 12 |
| 3 | 0 | 12 |

For a clean reset, use a fresh Guided Demo environment, or remove both demo business tables **and** the scoped `METADATA_SOURCE_OBSERVATION` history for the `inventory_movements` → `inventory_movements_incremental` relationship through an approved metadata-maintenance process. Source Observation stores the source-state history used by Incremental execution, while successful watermark progress is stored on the governed target. Then set `DEMO_DAY = 1` and rerun from the top.
